# 08C 用同一个助手导读 Hello-Agents

Hello-Agents 在这里是输入资料，不是本课程的主教材或运行依赖。我们固定一个提交版本，只选择 README、第四节说明和相关源码，学习同样的“地图、检索、原文、解释、核验”过程。

默认读取来源清单和已保存的结果元信息，不下载、不执行第三方代码，不把原文复制进课程仓库。

**本实践的交付位置**：只复用“资料地图—检索—原文—解释—核验”的流程，不复用 Mini Guide 的坐标和结论；最终用固定提交、补证记录和源码分支核验说明迁移是否真的完成。

**贯穿案例**

把同一导读流程迁移到固定提交的 Hello-Agents 仓库，问题缩到 ReAct 循环的正常停止与预算停止。先读来源清单和保存的真实答复，再用固定版本源码核对 `Finish`、工具错误和停止分支；若原文没在本机，清楚标出尚未复核的范围，不把来源清单当全文。


## 学习路线与配套课件

本 Notebook 可以独立从头运行。先读定义和输入，预测结果，再执行并解释差异。小算例帮助理解机制；真实训练、工具执行与保存的 API 记录会分别标明来源。

本节重点：从固定源码核对停止与错误分支。

课件页：L08.05-S02、L08.05-S03。

沿原有课程单元顺序完成实验；新增对照放在相关代码后。练习先自行作答，展开参考分析后再比较。

In [1]:
from pathlib import Path
import sys, json, hashlib, tempfile, copy, subprocess, inspect
ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "00-资料库使用说明.md").exists())
PROJECT = ROOT / "08 综合实践一 资料与仓库导读/代码/学习助手"
sys.path.insert(0,str(PROJECT / "src"))
from corpus import Corpus
from study_runtime import StudyRuntime, Harness, citation_check
from checkpoint_store import CheckpointStore
from IPython.display import display, Code
print("自编资料默认离线可用；导入不执行仓库代码。")


自编资料默认离线可用；导入不执行仓库代码。


## 1. 固定版本，缩小范围

仓库分支会变化，引用的文件和行号要能回到同一个版本。输入清单记录提交 SHA、路径、Git blob 标识和 SHA-256。

先选择范围再导入，比一开始吞下所有资料更容易讲清楚。版本固定说明我们读的是哪份材料，不意味着其内容一定正确或与当前所有软件兼容。


In [2]:
manifest=json.loads((ROOT/"08 综合实践一 资料与仓库导读/数据/hello-agents-input.json").read_text(encoding='utf-8'))
print("仓库:",manifest["repository"],"提交:",manifest["revision"])
for item in manifest["files"]:print(item["role"],item["path"],item["sha256"][:12])
print("许可证文件头:",manifest["license_header"])
# 宣传联动与复制内容的授权范围不是同一件事。


仓库: datawhalechina/hello-agents 提交: 4f7682ceafe573d07cd8a7d0b89908500e83227d
reading README.md e02fb30e4b7c
reading code/chapter4/ReAct.py bf792fddcd8f
reading code/chapter4/llm_client.py c8c308ec0d9b
reading code/chapter4/tools.py 8b7b10c4713b
reading docs/chapter4/第四章 智能体经典范式构建.md 3f02e84fcf75
license LICENSE.txt 1349a4b61484
许可证文件头: Attribution-NonCommercial-ShareAlike 4.0 International


## 2. 可选获取原文

下面展示完整下载函数。它只访问清单里的固定 URL，下载后同时校验 blob 和 SHA-256；没有 pip install、import 仓库模块或运行入口。

需要看原文时再显式打开开关。缓存保存在 .local，默认不进入 Git。大 PDF 可改用同一导入器，但先缩小到教学所需页；当前没有扫描图 OCR。


In [3]:
from fetch_reference import fetch
display(Code(inspect.getsource(fetch),language="python"))


def fetch():
    manifest=json.loads(MANIFEST.read_text(encoding='utf-8'))
    base=ROOT/'.local/external-materials/hello-agents'/manifest['revision']
    reading=base/'reading';reading.mkdir(parents=True,exist_ok=True)
    for item in manifest['files']:
        target=(base if item['role']=='license' else reading)/item['path']
        target.parent.mkdir(parents=True,exist_ok=True)
        data=target.read_bytes() if target.exists() else urllib.request.urlopen(item['url'],timeout=40).read(1_000_001)
        if len(data)>1_000_000:raise ValueError('下载超出文本上限')
        blob=hashlib.sha1(b'blob '+str(len(data)).encode()+b'\0'+data).hexdigest()
        if blob!=item['git_blob_sha1']:raise ValueError('文件与固定提交清单不一致')
        if 'sha256' in item and hashlib.sha256(data).hexdigest()!=item['sha256']:raise ValueError('内容哈希不一致')
        target.write_bytes(data)
    print(reading)
    return reading

In [4]:
DOWNLOAD_REFERENCE = False
if DOWNLOAD_REFERENCE:
    directory=fetch()
    optional_corpus=Corpus(directory)
    print("文档数:",len(optional_corpus.documents),"资料块:",len(optional_corpus.chunks))
    for hit in optional_corpus.search("ReActAgent max_steps Observation",3)["hits"]:
        print(hit["id"],hit["location"])
else:
    print("未下载；用保存的来源清单与真实导读元信息继续学习。")


未下载；用保存的来源清单与真实导读元信息继续学习。


## 3. 同一方法，读取不同仓库

真实运行目标仅限 ReActAgent.run 的停止条件和 Observation 如何进入下一轮。工具和 Harness 保持相同，换的是资料目录与任务。

完整含原文轨迹保留在受控本地缓存；随课只保存回答、来源位置、版本和读取产物哈希。下方不把“下载过”当作“已经运行了仓库的 Agent”。


In [5]:
record=json.loads((PROJECT/"results/deepseek-study.json").read_text(encoding='utf-8'))
case=next(c for c in record["cases"] if c["name"]=="hello-agents")
print("本次运行模型:",record["model"],"记录时间:",record["recorded_at"])
print("未来重跑规则: 先查询当前账号的可用模型列表。")
print("固定版本:",case["revision"],"状态:",case["status"])
print("请求 / 工具:",case["model_requests"],case["tool_calls"])
print(case["answer"])
print(json.dumps(case["citations"],ensure_ascii=False,indent=2))
assert case["revision"]==manifest["revision"]


本次运行模型: deepseek-flash 记录时间: 2026-09-24T05:52:33.115397+00:00
未来重跑规则: 先查询当前账号的可用模型列表。
固定版本: 4f7682ceafe573d07cd8a7d0b89908500e83227d 状态: final_answer
请求 / 工具: 4 6
**两步导读（ReActAgent.run）**

1. **看停止条件**：读 `ReAct.py` 的 while 循环。循环有步数上限，超限即终止；同时在 Action 执行处先判断是否为 `Finish`，是则结束本轮"思考-行动"过程 [0ca1c103c44d7b81]。

2. **看 Observation 如何回到下一轮**：工具执行得到结果后，把本轮的 Action 与 Observation 都追加进 history，成为下一轮提示的新上下文，从而形成闭环 [d8b7c77555db9440]；无效 Action 格式会写入提示性 Observation 后继续 [157c2f6857c66ed9]。

检查点：读完能说出"退出靠步数上限 + Finish"，以及"Observation 进 history"这两处即可。
{
  "has_citations": true,
  "all_citations_read": false,
  "locations": {},
  "unknown_or_unread": [
    "0ca1c103c44d7b81",
    "d8b7c77555db9440",
    "157c2f6857c66ed9"
  ],
  "semantic_support": "requires_source_review"
}


## 4. 保留原答复，补读全文后再核验

原始回答有引用只来自预览。补证时增加短文件全文读取，沿同一任务继续。本次预跑在原七轮预算内得到回答；另有一次历史运行达到预算，操作者明确增加一轮后才得到更正。两次运行 ID 不同，不能拼成一条轨迹。

即使引用能够读回，仍需按源码核对“解析不出 Action”和“工具名/入参格式无效”等分支；前者 break，后者追加错误 Observation 后 continue。当前模型回答若把两者混同，应保留原答复并标记这一事实错误。

In [6]:
repair=json.loads((PROJECT/"results/deepseek-study-repair.json").read_text(encoding='utf-8'))
reference=next(c for c in repair["cases"] if c["name"]=="hello-agents")
final=json.loads((PROJECT/"results/deepseek-reference-extension.json").read_text(encoding='utf-8'))
print("补证后状态:",reference["status"],reference["after"])
print("当前补证回答:",reference["answer"])
print("当前引用读取检查:",reference["citations"]["all_citations_read"])
print("另一轮历史扩额记录:",final["before"],final["after"])
print("两条记录属于同一 run:",final["run_id"]==reference["run_id"])
assert final["after"]["turns"]==final["before"]["turns"]+1
print("历史扩额后的引用读取检查:",final["citations"]["all_citations_read"])

补证后状态: final_answer {'turns': 7, 'tool_calls': 8}
当前补证回答: **两步导读（ReAct.py，完整读过 [720fff051038a9c3]）**

1. **停止有四条路**：正常是走到 `Finish` 直接返回答案；其次是 LLM 返回空、Action 解析失败、以及循环用满 `max_steps`（默认 5）——后三条都以终止收场并可能返回 `None` [720fff051038a9c3]。

2. **Observation 进下一轮**：每轮把本轮的 Action 文本和工具返回的 Observation 依次追加进 history；下一轮把 history 拼进提示再次询问模型。若 Action 格式无效，只写入一条提示性 Observation，然后继续循环 [720fff051038a9c3]。

检查点：能分清"Finish 是成功出口，其余是异常出口"，并说出 history 是唯一的跨轮记忆 [720fff051038a9c3]。
当前引用读取检查: True
另一轮历史扩额记录: {'turns': 7, 'tool_calls': 7, 'limit': 7} {'turns': 8, 'tool_calls': 7, 'limit': 8}
两条记录属于同一 run: False
历史扩额后的引用读取检查: True


### 从固定源码核对停止与错误分支

配套课件：L08.05-S02、L08.05-S03。

补读后有回答，不等于所有停止分支都被讲对。这里读取已按固定源码逐项核对的静态审阅表：版本与文件 SHA-256 对齐随课清单，位置可回到原源码。特别区分模型异常、内置 search 的异常和其他工具未捕获异常，它们不一定走同一个分支。

In [7]:
case_review = json.loads((ROOT / "08 综合实践一 资料与仓库导读/数据/hello-agents-branches.json").read_text(encoding="utf-8"))
assert case_review["revision"] == manifest["revision"]
case_hashes = {item["path"]: item["sha256"] for item in manifest["files"]}
assert all(case_hashes[path] == sha for path, sha in case_review["sources"].items())
for case_row in case_review["branches"]:
    print("条件:", case_row["condition"])
    print("行为:", case_row["behavior"])
    print("位置:", case_row["location"], "\n")
print("核验方式:", case_review["method"])
print("同一任务继续:", final["run_id"] == reference["run_id"])
assert final["after"]["turns"] == final["before"]["turns"] + 1


条件: Action 以 Finish 开头
行为: 解析方括号内容并立即 return；格式不完整时解析结果可能为空串
位置: ReAct.py:54-58,88-90 

条件: 响应为空或未解析出 Action
行为: break 后到达公共尾部并返回 None；并非一定用尽步数
位置: ReAct.py:47-52,72-73 

条件: 普通 Action 的工具名或输入格式无效
行为: 追加错误 Observation 后 continue，下一轮仍会消耗步数
位置: ReAct.py:60-62 

条件: 工具未注册
行为: 生成错误字符串并追加 Action/Observation，循环可以继续
位置: ReAct.py:65-70 

条件: while 条件不再满足 max_steps
行为: 退出循环，打印尾部日志并返回 None
位置: ReAct.py:37-38,72-73 

条件: 模型请求或流式读取发生异常
行为: think 的 try/except 返回 None，run 随后走空响应 break 分支；初始化异常不在该 try 内
位置: llm_client.py:23-26,33-55; ReAct.py:46-48 

条件: 内置 search 在其 try 内发生异常
行为: search 返回错误字符串，循环把它当 Observation 保存
位置: tools.py:15-49; ReAct.py:65-70 

条件: 其他注册工具抛出未捕获异常
行为: run 的工具调用周围没有 try/except，异常向调用者传播；不能笼统称所有工具错误都会继续
位置: ReAct.py:65-70 

核验方式: static_source_review_after_sha256_verification; upstream code not executed
同一任务继续: False


**结果解读**

Finish 分支直接返回；空响应或缺 Action 会先 break 再返回 None；达到预算也返回 None。尾部“达到最大步数”日志可能在提前 break 后出现，不能单靠日志归因。内置 search 会把其 try 内的异常转为字符串，但 run 没有兜底捕获任意注册工具的异常。以上是固定源码的静态核验，不是运行上游服务的结果。

**小练习**

挑选“其他工具抛出异常”与“工具未注册”两个分支，解释为什么一个传播异常，另一个还可继续。若要给助手增加错误恢复，你会在哪个调用边界实现？

<details><summary>完成后展开参考分析</summary>

未注册时程序构造错误字符串并保存到 history；已注册函数在调用时抛出未捕获异常，会离开 run。可以在实际工具执行边界设计明确的异常分类、观察返回和重试规则，保留原错误，不应把所有异常都当作可恢复。

</details>

公式与实现来源见 [配套素材来源](../../资料来源.md)。

### 按源码逐项核验

- 无响应，或没有解析出 Action：break。
- Action 以 Finish 开头：直接 return 最终文本。
- 工具名/参数格式无效：追加错误 Observation，continue。
- 有效普通工具动作：把 Action 和工具结果追加到 history；工具不存在时，结果是错误字符串，也追加。
- 每轮开头把 history 拼入下一次提示；达到 max_steps 时退出循环并返回 None。
- 末尾的“达到最大步数”日志也可能在前面的 break 后出现，不能只凭这一行日志判断实际停止原因。

这是读代码后的分支核验，不代表运行过上游 Agent，也不保证随机模型一定正确停止。


## 练习：迁移到你选的一份资料

先写一个明确的学习目标和两项完成条件，再列出想读的文件、固定版本和待核查问题。不要直接输入“帮我学完整个仓库”。

实际迁移实现将在第九节的另一个自编仓库里完整运行；本单元先检查输入契约。


In [8]:
migration={"goal":"解释一个循环的正常停止与预算停止", "scope":["明确的入口文件"],
    "version":"填写固定提交或文档哈希", "success":["指出实际停止分支","给出一个可回源码验证的问题"]}
print(migration)


{'goal': '解释一个循环的正常停止与预算停止', 'scope': ['明确的入口文件'], 'version': '填写固定提交或文档哈希', 'success': ['指出实际停止分支', '给出一个可回源码验证的问题']}


In [9]:
assert len(migration["success"])==2
print("正式迁移前，用实际文件与版本替换计划字段；本例是设计练习，不冒充已导入的新仓库。")


正式迁移前，用实际文件与版本替换计划字段；本例是设计练习，不冒充已导入的新仓库。


## 交付：跨仓库迁移记录

记录固定版本、缩小后的资料范围、原回答的不足、补证与扩额结果、静态分支核验和未来模型选择规则。这是可迁移的工作过程，不是可直接复用的历史结论。


In [10]:
delivery_dir = ROOT / ".local/student-runs/l08"
delivery_dir.mkdir(parents=True, exist_ok=True)
transfer_report = {
    "artifact_type": "cross_repository_transfer_report",
    "repository": manifest["repository"],
    "revision": manifest["revision"],
    "selected_files": [{"role": item["role"], "path": item["path"], "sha256": item["sha256"]} for item in manifest["files"]],
    "run_model": record["model"],
    "recorded_at": record["recorded_at"],
    "initial_status": case["status"],
    "repaired_status": reference["status"],
    "same_run_after_extension": final["run_id"] == reference["run_id"],
    "turn_budget_before": final["before"]["turns"],
    "turn_budget_after": final["after"]["turns"],
    "all_citations_read": final["citations"]["all_citations_read"],
    "migration_contract": migration,
    "model_selection_note": "Before a live rerun, list models available to the current account and choose a supported one.",
}
(delivery_dir / "cross-repository-transfer-report.json").write_text(
    json.dumps(transfer_report, ensure_ascii=False, indent=2), encoding="utf-8"
)
(delivery_dir / "cross-repository-transfer-report.md").write_text(
    "# 跨仓库迁移记录\n\n"
    f"- 仓库：{transfer_report['repository']}\n"
    f"- 固定提交：`{transfer_report['revision']}`\n"
    f"- 本次模型：`{transfer_report['run_model']}`\n"
    f"- 同一任务扩额继续：{transfer_report['same_run_after_extension']}\n"
    f"- 引用全部来自已读证据：{transfer_report['all_citations_read']}\n\n"
    "> 换仓库后必须重建资料身份与证据坐标；未来重跑前先查询当前账号可用模型。\n",
    encoding="utf-8",
)
print("已保存:", delivery_dir / "cross-repository-transfer-report.json")


已保存: .local\student-runs\l08\cross-repository-transfer-report.json


## 本节总结

学习助手围绕小目标组织资料、动作与自检；PDF 和源码都保留可回查位置；同一运行系统可以接入另一个资料目录。

Hello-Agents 是本次输入示例，迁移方法适用于更多明确授权的资料。下一节加入学生尝试、实际执行反馈和继续学习记录。

来源：[固定版本仓库](https://github.com/datawhalechina/hello-agents/tree/4f7682ceafe573d07cd8a7d0b89908500e83227d)、随课 references/hello-agents-input.json。
